In [ ]:
# --- repo-root guard ---
# Notebooks live in notebooks/, but all data/code paths are relative to the repo root.
# If launched with the working directory set to notebooks/, step up one level so that
# 'src/', 'data/', 'pride_data/' and 'figures/' resolve correctly. Idempotent / no-op at root.
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / 'src').exists() and (_cwd.parent / 'src').exists():
    os.chdir(_cwd.parent)
print('working directory:', pathlib.Path.cwd())


In [1]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os, math
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import importlib, core
importlib.reload(core)
from core import (count_sites_per_sample_ptm_report, process_ptm_site_report,
                  calculate_dilution_linearity, _hex_to_rgba)

In [2]:
# Shape-number gradient (light->dark across 100..1500 shapes)
SHAPE_ORDER = [100, 200, 300, 500, 1000, 1500]
SHAPE_COLORS = ['#EEA69B', '#E78373', '#E1604C', '#DB452E', '#B3321E', '#641C11']
SITE_COLORS  = ['#ef745c', '#d06257', '#b15052', '#923e4d', '#722b47', '#34073d']
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# Data upload

In [3]:
# phosphoDVP shape-number dilution series: laser-microdissected mouse-brain "shapes"
# (100-1500), wide PTM Site Reports (Class I per-run), n=4 replicates each.
RAW_DIR = Path('pride_data/analysis_data/revision/figure5')
shapes = {n: pd.read_csv(next(RAW_DIR.glob(f'*DVPP_{n}shapes_Report.tsv')), sep='\t', low_memory=False)
          for n in SHAPE_ORDER}
print('shape datasets:', list(shapes))

shape datasets: [100, 200, 300, 500, 1000, 1500]


# Supplementary Figure 5a

Phosphopeptide-precursor depth as a function of the number of microdissected shapes (phosphoDVP) — the precursor-level mirror of Fig. 5b (Class I sites), same box convention and shape-gradient colours. Phosphopeptide precursors = unique phosphorylated precursors (`EG.PrecursorId`), localization-independent, per Bekker-Jensen et al. 2020 (n = 4 per shape number).

In [4]:
# Supplementary Figure 5a — phosphopeptide-precursor depth vs number of microdissected shapes
# (precursor-level mirror of Fig. 5b). Phosphopeptide precursors = unique phosphorylated precursors
# (EG.PrecursorId), localization-independent, per Bekker-Jensen 2020. Per RUN, decoys removed (n=4).
import os, glob
import numpy as np, pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR = r'pride_data/analysis_data/figure5'
SHAPE_ORDER = [100, 200, 300, 500, 1000, 1500]
SHAPE_COLORS = ['#EEA69B', '#E78373', '#E1604C', '#DB452E', '#B3321E', '#641C11']

def count_phosphoprecursors_per_run(path):
    """Unique phosphorylated precursors (EG.PrecursorId) per run; decoys removed."""
    df = pd.read_csv(path, sep='\t',
                     usecols=['R.FileName', 'EG.ModifiedSequence', 'EG.PrecursorId', 'EG.IsDecoy'],
                     low_memory=False)
    decoy = (df['EG.IsDecoy'].astype(str).str.lower().isin(['true', '1'])
             if df['EG.IsDecoy'].dtype != bool else df['EG.IsDecoy'])
    df = df[~decoy]
    df = df[df['EG.ModifiedSequence'].astype(str).str.contains('Phospho', case=False, na=False)]
    return df.groupby('R.FileName')['EG.PrecursorId'].nunique().tolist()

def find_prec(n):
    hits = glob.glob(os.path.join(PREC_DIR, f'*DVPP_{n}shapes_out_Report_phosphopeptides*.tsv'))
    return hits[0] if hits else None

prec_counts = {}
for n in SHAPE_ORDER:
    p = find_prec(n)
    if p:
        prec_counts[n] = count_phosphoprecursors_per_run(p)

rows = [n for n in SHAPE_ORDER if n in prec_counts]
summary = pd.DataFrame({'mean':   [int(np.mean(prec_counts[n])) for n in rows],
                        'median': [int(np.median(prec_counts[n])) for n in rows],
                        'CV%':    [round(100*np.std(prec_counts[n], ddof=1)/np.mean(prec_counts[n]), 1) for n in rows]},
                       index=rows)
summary.index.name = 'shapes'
print(summary.to_string())

fig = go.Figure()
for i, n in enumerate(SHAPE_ORDER):
    if n not in prec_counts:
        continue
    ys = prec_counts[n]
    fig.add_trace(go.Box(y=ys, x=[str(n)] * len(ys), name=str(n), boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=8, color=SHAPE_COLORS[i], line=dict(width=0.5, color='black')),
        line=dict(color=SHAPE_COLORS[i], width=1.5), fillcolor=_hex_to_rgba(SHAPE_COLORS[i], 0.2),
        showlegend=False))
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='Number of shapes', yaxis_title='Phosphopeptide precursors')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in SHAPE_ORDER])
fig.update_yaxes(rangemode='tozero')
fig.show()
fig.write_image(r'figures/figure5/suppl_figure5a.pdf', width=600, height=600)


         mean  median   CV%
shapes                     
100       857     840   4.5
200      3309    3308   1.1
300      4813    4817   0.9
500      8210    8809  16.9
1000    13965   13964   0.8
1500    18823   19030   5.9


# Supplementary Figure 5b
Estimated protein input per shape-number. Because these are fresh-frozen mouse-brain shapes, input is estimated by mapping each shape group's per-sample Class I depth onto the **fresh-frozen Class I dilution calibration** (Fig. 4a / Suppl. Fig. 3a; Michaelis–Menten fit of Class I sites vs known protein input), then inverting `input = Km·count/(Vmax − count)`. This replaces the legacy identification-count calibration (which was fit on unfiltered counts and is invalid for Class I data).

In [5]:
# Suppl 5b - estimated protein input per shape, via the fresh-frozen Class I MM dilution calibration.
from scipy.optimize import curve_fit
FIG4_DIR  = Path('pride_data/analysis_data/revision/figure4')
FF_INPUTS = [10, 20, 50, 100, 200, 500, 1000]

def ff_report(ng):
    if ng == 1000:
        return next(FIG4_DIR.glob('*dilser_FF_1000ng_repeat_Report.tsv'))
    return next(FIG4_DIR.glob(f'*dilser_FF_{ng}ng_Report.tsv'))

# FF fresh-frozen Class I per-sample counts vs known protein input -> Michaelis-Menten fit
ff_x, ff_y = [], []
for ng in FF_INPUTS:
    for v in count_sites_per_sample_ptm_report(pd.read_csv(ff_report(ng), sep='\t', low_memory=False)).values():
        ff_x.append(ng); ff_y.append(v)
def michaelis_menten(x, Vmax, Km):
    return Vmax * x / (Km + x)
(Vmax, Km), _ = curve_fit(michaelis_menten, np.array(ff_x, float), np.array(ff_y, float),
                          p0=[max(ff_y) * 1.5, 200], maxfev=10000)
print(f'FF Class I dilution calibration: Vmax={Vmax:.0f} sites, Km={Km:.0f} ng')

# per-sample Class I depth of each shape group -> invert the MM calibration -> estimated ng input
shape_depth = {n: float(np.mean(list(count_sites_per_sample_ptm_report(shapes[n]).values()))) for n in SHAPE_ORDER}
def inv_mm(c):
    return Km * c / (Vmax - c) if c < Vmax else np.nan
est_input = [inv_mm(shape_depth[n]) for n in SHAPE_ORDER]
print(pd.DataFrame({'per_sample_ClassI': [int(shape_depth[n]) for n in SHAPE_ORDER],
                    'est_input_ng': np.round(est_input, 1)}, index=SHAPE_ORDER).to_string())

fig = go.Figure()
fig.add_trace(go.Scatter(x=SHAPE_ORDER, y=est_input, mode='lines+markers',
                         marker=dict(size=14, color='#ef755d'), line=dict(color='#ef755d')))
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='Number of shapes', yaxis_title='Estimated protein input (ng, FF-calibrated)')
fig.update_yaxes(rangemode='tozero')
fig.show()
#fig.write_image(r'figures/figure5/suppl_figure5b.pdf', width=600, height=600)

FF Class I dilution calibration: Vmax=10749 sites, Km=223 ng
      per_sample_ClassI  est_input_ng
100                 244           5.2
200                 670          14.8
300                 937          21.3
500                1545          37.4
1000               2565          69.9
1500               3617         113.1


# Supplementary Figure 5c
Coefficient of variation of Class I phosphosite intensities across replicates, per shape number. Per project policy CV keeps multiplicity (per-feature quantitative metric); linear-space CV requiring all replicates valid.

In [6]:
# Suppl 5c - replicate CV of Class I intensities per shape number.
cv_by_shape = {}
for n in SHAPE_ORDER:
    sd = process_ptm_site_report(shapes[n], cutoff=0.75)['site_data']
    cols = [c for c in sd.columns if c not in PROC_META]
    lin = np.power(2.0, sd[cols])
    cv = (lin.std(axis=1) / lin.mean(axis=1)).where(lin.notna().sum(axis=1) >= len(cols))
    cv_by_shape[n] = cv.dropna()
allcv = np.concatenate([v.values for v in cv_by_shape.values()])
print('median CV (all shapes): %.3f' % np.nanmedian(allcv))
for n in SHAPE_ORDER:
    print(f'  {n:>4} shapes: n={len(cv_by_shape[n]):>5}  median CV={np.nanmedian(cv_by_shape[n]):.3f}')

fig = go.Figure()
for i, n in enumerate(SHAPE_ORDER):
    fig.add_trace(go.Box(y=cv_by_shape[n], x=[str(n)]*len(cv_by_shape[n]), name=str(n),
        marker_color=SHAPE_COLORS[i], line=dict(color=SHAPE_COLORS[i]), boxpoints='outliers',
        marker=dict(size=3, opacity=0.4), showlegend=False))
fig.add_hline(y=np.nanmedian(allcv), line={'dash': 'dash', 'width': 2, 'color': 'black'})
fig.update_layout(width=900, height=500, template='plotly_white',
                  xaxis_title='Number of shapes', yaxis_title='Coefficient of variation')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in SHAPE_ORDER])
fig.update_yaxes(rangemode='tozero')
fig.show()
#fig.write_image(r'figures/figure5/suppl_figure5c.pdf', width=900, height=500)

Dropped 31 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 1,002 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 406 → 401.
Final: 401 sites × 4 samples.
Dropped 261 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 2,789 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,120 → 1,112.
Final: 1,112 sites × 4 samples.
Dropped 232 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 3,910 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,542 → 1,539.
Final: 1,539 sites × 4 samples.
Dropped 926 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75

# Supplementary Figure 5d
AlphaQuant differential phosphorylation between excitatory and inhibitory cortical neurons (act_cortex vs inh_cortex), ported from Figure5_v00. The heavy AlphaQuant run is gated behind `RERUN_AQ` (existing results are read by default); the volcano shows significant sites (green / FDR) up in excitatory (red) or inhibitory (blue).

In [7]:
# Suppl 5d - AlphaQuant act_cortex vs inh_cortex (ported from Figure5_v00).
import os
AQ_INPUT  = r'pride_data/analysis_data/figure5/pDVP_phospho_neurons_AQ.tsv'
AQ_OUTDIR = r'pride_data/analysis_data/figure5/aq_out'
RESULTS   = os.path.join(AQ_OUTDIR, 'act_cortex_VS_inh_cortex.results.tsv')
RERUN_AQ  = False     # set True to regenerate AlphaQuant results (heavy; writes to AQ_OUTDIR)

if RERUN_AQ:
    import sys
    sys.path.insert(1, r'src/alphaquant')
    import alphaquant.run_pipeline as aq_pipeline
    cs = pd.read_csv(r'data/pDVP_neurons_conditionSetup.tsv', sep='\t')
    cs['Condition'] = cs['Condition'].str[:-1]                       # act_cortex1 -> act_cortex
    cs['Sample'] = cs['Run Label'].str.split('.').str[0]
    s2c = dict(zip(cs['Sample'], cs['Condition']))
    df = pd.read_csv(AQ_INPUT, sep='\t')
    df1 = df[df['R.FileName'].isin(cs['Sample'])].copy()
    rlabel_to_cond = (df1[['R.Label', 'R.FileName']].drop_duplicates('R.Label')
                      .assign(cond=lambda d: d['R.FileName'].map(s2c)))
    samplemap = pd.DataFrame({'sample': rlabel_to_cond['R.Label'], 'condition': rlabel_to_cond['cond']})
    os.makedirs(AQ_OUTDIR, exist_ok=True)
    sm_path = os.path.join(AQ_OUTDIR, 'samplemap_phospho.tsv'); samplemap.to_csv(sm_path, sep='\t', index=False)
    aq_in = os.path.join(AQ_OUTDIR, 'pDVP_phospho_neurons_AQ.tsv'); df1.to_csv(aq_in, sep='\t')
    aq_pipeline.run_pipeline(input_file=aq_in, samplemap_file=sm_path, results_dir=AQ_OUTDIR,
                             condpairs_list=[('act_cortex', 'inh_cortex')], perform_ptm_mapping=True,
                             modification_type='[Phospho (STY)]', organism='mouse', volcano_fcthresh=0.5)
print('reading AlphaQuant results:', RESULTS)

reading AlphaQuant results: pride_data/analysis_data/figure5/aq_out\act_cortex_VS_inh_cortex.results.tsv


In [8]:
# Suppl 5d volcano - act_cortex vs inh_cortex.
ttest_res = pd.read_csv(RESULTS, sep='\t')
ID = []
for _, row in ttest_res.iterrows():
    if (row['log2fc'] > 0.5) and (row['color'] == 'green'):
        ID.append('up')
    elif (row['log2fc'] < -0.5) and (row['color'] == 'green'):
        ID.append('down')
    else:
        ID.append('none')
ttest_res['ID'] = ID
print('act_cortex vs inh_cortex significant:',
      {k: ID.count(k) for k in ('up', 'down', 'none')})

fig = px.scatter(ttest_res, x='log2fc', y='-log10(fdr)', color='ID',
                 color_discrete_map={'none': '#d5d9d8', 'up': '#d3321d', 'down': '#5257e5'})
fig.update_traces(marker=dict(size=13, line=dict(width=0.5)))
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_xaxes(title='log2FC (act_cortex vs inh_cortex)'); fig.update_yaxes(title='-log10(FDR)')
fig.show()
# fig.write_image(r'figures/figure5/suppl_figure5d.pdf', width=600, height=600)

act_cortex vs inh_cortex significant: {'up': 73, 'down': 91, 'none': 3852}


In [9]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 5   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

_try(lambda: dump_panel(pd.DataFrame([{"Number of shapes":n,"Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for n in sorted(prec_counts) for i,v in enumerate(prec_counts[n])]),f"Suppl Figure {SFIG}a"),f"Suppl Figure {SFIG}a")

_try(lambda: dump_panel(pd.DataFrame({"Number of shapes":list(SHAPE_ORDER),
    "per_sample_ClassI":[int(shape_depth[n]) for n in SHAPE_ORDER],
    "Estimated Input":[est_input[i] for i in range(len(SHAPE_ORDER))]}),
    f"Suppl Figure {SFIG}b"),f"Suppl Figure {SFIG}b")
_try(lambda: dump_panel(pd.concat({f"{n}shapes":cv_by_shape[n].reset_index(drop=True) for n in SHAPE_ORDER},axis=1),
    f"Suppl Figure {SFIG}c"),f"Suppl Figure {SFIG}c")
_try(lambda: dump_panel(ttest_res,f"Suppl Figure {SFIG}d"),f"Suppl Figure {SFIG}d")
print(f"Suppl Figure {SFIG} export done.")


  [MetaInfo] wrote 'Suppl Figure 5a'  (24 rows x 3 cols)
  [MetaInfo] wrote 'Suppl Figure 5b'  (6 rows x 3 cols)
  [MetaInfo] wrote 'Suppl Figure 5c'  (1969 rows x 6 cols)
  [MetaInfo] wrote 'Suppl Figure 5d'  (4016 rows x 14 cols)
Suppl Figure 5 export done.
